# Using Codex to prepare simulations with DeepTrack2

This notebook is an example of how one could prompt an AI agent to generate code for preparing simulations with DeepTrack2. 
It defines the a spherical particle and an
optical configuration. You can use your own Codex CLI to
write one reusable simulation.py, and then execute that simulator on the
requested particle and on a second particle.

The default request is fluorescence. You can change or edit MODALITY
to try brightfield, darkfield, iSCAT, or inline
holography. ARM selects either the baseline request or the
DeepTrack2 request.

Every run is stored below deeptrack_with_agents/.

## Before running

1. Use a Python environment with NumPy and Matplotlib. For ARM =
   "deeptrack2", use an environment in which DeepTrack2 is installed.
2. Install the Codex CLI if needed, then sign in from a terminal with your own
   account. The official Codex CLI guide covers installation, sign-in, and
   codex exec: https://learn.chatgpt.com/docs/codex/cli
3. Run the setup and configuration cells first. Set RUN_CODEX = True only after reviewing the prompt and output
   directory.

In [ ]:
from __future__ import annotations

import copy
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


MODALITIES = ("fluorescence", "brightfield", "darkfield", "iscat", "holography")

# Edit these three values
MODALITY = "fluorescence"
ARM = "deeptrack2"  # choose "baseline" or "deeptrack2"
RUN_LABEL = "run_00"  # use a new label to keep an earlier run

NOTEBOOK_ROOT = Path.cwd().resolve()
REPRO_ROOT = NOTEBOOK_ROOT / "deeptrack_with_agents"
REPRO_ROOT.mkdir(parents=True, exist_ok=True)

assert MODALITY in MODALITIES
assert ARM in {"baseline", "deeptrack2"}
print(f"working directory: {NOTEBOOK_ROOT}")
print(f"selected task: {MODALITY} | arm: {ARM} | run: {RUN_LABEL}")

## 1. Define the requested particle and optical configuration

Positions are image-pixel coordinates, axial position and radius are in micrometres,
refractive index is [real, imaginary], and fluorescence intensity is an
arbitrary simulator intensity parameter.

The second particle is only a reusability check. It is not shown to Codex
while the simulator is generated.

In [ ]:
particle = {
    "position_xy": [67.0, 59.0],
    "axial_position_um": 1.5,
    "radius_um": 0.45,
    "refractive_index": [1.45, 0.01],
    "fluorescence_intensity": 10.0,
}

reuse_particle = {
    "position_xy": [39.0, 82.0],
    "axial_position_um": 2.3,
    "radius_um": 0.72,
    "refractive_index": [1.51, 0.02],
    "fluorescence_intensity": 7.0,
}

BASE_CONFIG = {
    "seed": 123,
    "image_size": 128,
    "camera_resolution_um": 1.0,
    "magnification": 10.0,
    "numerical_aperture": 0.8,
    "medium_refractive_index": 1.33,
    "fluorescence_wavelength_nm": 600.0,
    "coherent_wavelength_nm": 532.0,
    "padding_px": 32,
    "iscat_illumination_angle_rad": float(np.pi),
    "iscat_amplitude_factor": 1.0,
}


def config_for(modality: str) -> dict:
    config = copy.deepcopy(BASE_CONFIG)
    config["modality"] = modality
    config["particle"] = copy.deepcopy(particle)
    return config


print(json.dumps({"particle": particle, "reuse_particle": reuse_particle}, indent=2))

## 2. Choose or edit the task prompt

The task text below is the prompt that will be sent to Codex. You can edit it to change the request.

In [ ]:
MODALITY_TASKS = {
    "fluorescence": (
        "Simulate physically motivated incoherent fluorescence image formation "
        "for a uniformly fluorescent spherical particle using the configured "
        "fluorescence wavelength and numerical aperture."
    ),
    "brightfield": (
        "Simulate brightfield imaging of a dielectric spherical particle using "
        "physically motivated coherent sphere scattering and image formation. "
        "Do not draw a Gaussian, ring, halo, Mexican-hat, or other hand-made "
        "brightfield template."
    ),
    "darkfield": (
        "Simulate darkfield imaging of a dielectric spherical particle using "
        "physically motivated sphere scattering and image formation in which "
        "the unscattered illumination is rejected. Do not draw a modality-like "
        "template."
    ),
    "iscat": (
        "Simulate interferometric scattering microscopy (iSCAT) of a dielectric "
        "spherical particle. The detector image must arise from interference "
        "between an appropriate reference field and the coherently scattered "
        "field of the sphere. Do not draw rings or halos by hand."
    ),
    "holography": (
        "Simulate inline holography of a dielectric spherical particle using "
        "coherent sphere scattering, propagation, and interference with the "
        "reference illumination. Do not draw a hologram-like ring pattern by "
        "hand."
    ),
}

task_description = MODALITY_TASKS[MODALITY]
print(task_description)

In [ ]:
def build_task_text(modality: str, description: str) -> str:
    return textwrap.dedent(
        f"""
        # Task: reusable {modality} simulation pipeline

        Create only simulation.py.

        It must define:

        ~~~python
        def build_pipeline(config):
            ...
        ~~~

        The returned callable must accept a particle dictionary and return one
        two-dimensional NumPy-compatible image with shape
        [image_size, image_size].

        The particle dictionary contains:

        - position_xy: Cartesian [x, y] coordinates in image pixels;
        - axial_position_um;
        - radius_um;
        - refractive_index: [real, imaginary];
        - fluorescence_intensity.

        The optical parameters are provided in task_config.json.

        {description}

        The pipeline must be reusable: do not hard-code the supplied particle
        values into the implementation. It must work for a second particle
        dictionary with different position, radius, refractive index, axial
        position, and fluorescence intensity.

        Do not write plotting code, file I/O, argument parsing, training code,
        or benchmark logic.
        """
    ).strip() + "\n"


ARM_INSTRUCTIONS = {
    "baseline": (
        "Use standard scientific Python packages such as NumPy and SciPy as "
        "needed. Do not import DeepTrack2. Implement the requested reusable "
        "simulation pipeline in your own Python code."
    ),
    "deeptrack2": (
        "Use the installed DeepTrack2 package and its relevant scientific "
        "abstractions where appropriate. Compose the requested optical model "
        "with DeepTrack2 rather than reimplementing a library-provided model."
    ),
}


def build_codex_prompt(task_text: str, arm: str) -> str:
    return textwrap.dedent(
        f"""
        Complete one simulation-only scientific coding task in the current
        directory.

        Read task.md and task_config.json. Create exactly one file named
        simulation.py implementing the reusable interface requested by the
        task. You may run small local tests if useful, but do not write plotting,
        training, command-line, or result-file code. Do not install packages,
        use the network, or access files outside the current workspace.

        The reader will later import simulation.py, call
        build_pipeline(config), and evaluate the returned callable on both the
        requested particle and an unseen particle.

        Condition:
        {ARM_INSTRUCTIONS[arm]}

        Here is the complete task text:
        ---------------- TASK ----------------
        {task_text}
        -------------- END TASK --------------
        """
    ).strip() + "\n"


def prepare_run(
    modality: str,
    arm: str,
    run_label: str,
    task_description_override: str | None = None,
) -> dict:
    description = (
        MODALITY_TASKS[modality]
        if task_description_override is None
        else str(task_description_override)
    )
    config = config_for(modality)
    task_text = build_task_text(modality, description)
    prompt = build_codex_prompt(task_text, arm)
    run_dir = REPRO_ROOT / modality / arm / run_label
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "task.md").write_text(task_text, encoding="utf-8")
    (run_dir / "task_config.json").write_text(
        json.dumps(config, indent=2) + "\n", encoding="utf-8"
    )
    (run_dir / "codex_prompt.txt").write_text(prompt, encoding="utf-8")
    return {
        "run_dir": run_dir,
        "config": config,
        "task_text": task_text,
        "prompt": prompt,
    }


selected = prepare_run(
    MODALITY,
    ARM,
    RUN_LABEL,
    task_description_override=task_description,
)
RUN_DIR = selected["run_dir"]
config = selected["config"]
print(f"prepared run directory: {RUN_DIR}")
print(f"task prompt: {RUN_DIR / 'codex_prompt.txt'}")

## 3. Connect to your own Codex CLI

The next cell only defines a local subprocess wrapper. It does not send a
request until the following cell is run with RUN_CODEX = True.

The wrapper uses codex exec with a workspace-write sandbox and saves the JSONL
transcript and stderr beside the generated simulator. Authentication is
handled by the user's existing Codex CLI sign-in.

In [ ]:
CODEX_BIN = os.environ.get("CODEX_BIN", "codex")
CODEX_MODEL = os.environ.get("CODEX_MODEL") or None  # leave None for your CLI default
CODEX_TIMEOUT_SECONDS = 1800
ALLOW_OVERWRITE_EXISTING = False


def codex_command() -> list[str]:
    executable = shutil.which(CODEX_BIN)
    if executable is None:
        raise FileNotFoundError(
            "Codex CLI was not found. Install it, sign in from a terminal, "
            "or set CODEX_BIN to its executable path."
        )
    command = [
        executable,
        "exec",
        "--sandbox",
        "workspace-write",
        "--skip-git-repo-check",
        "--json",
    ]
    if CODEX_MODEL:
        command.extend(["-m", CODEX_MODEL])
    command.append("-")
    return command


def run_codex(prompt: str, run_dir: Path) -> subprocess.CompletedProcess[str]:
    simulation_path = run_dir / "simulation.py"
    if simulation_path.exists() and not ALLOW_OVERWRITE_EXISTING:
        raise FileExistsError(
            f"{simulation_path} already exists. Choose a new RUN_LABEL or set "
            "ALLOW_OVERWRITE_EXISTING = True after reviewing the file."
        )
    result = subprocess.run(
        codex_command(),
        cwd=run_dir,
        input=prompt,
        text=True,
        capture_output=True,
        timeout=CODEX_TIMEOUT_SECONDS,
        check=False,
    )
    (run_dir / "codex.stdout.jsonl").write_text(result.stdout, encoding="utf-8")
    (run_dir / "codex.stderr.txt").write_text(result.stderr, encoding="utf-8")
    if result.returncode != 0:
        raise RuntimeError(
            f"Codex exited with status {result.returncode}. See "
            f"{run_dir / 'codex.stderr.txt'}"
        )
    return result


print("Codex executable:", shutil.which(CODEX_BIN) or "not found")
print("Codex model:", CODEX_MODEL or "CLI default")

In [ ]:
RUN_CODEX = False  # set to True to run Codex after signing in from a terminal

if RUN_CODEX:
    codex_result = run_codex(selected["prompt"], RUN_DIR)
    print(f"Codex completed successfully in {RUN_DIR}")
    print(f"transcript: {RUN_DIR / 'codex.stdout.jsonl'}")
else:
    print("Set RUN_CODEX = True and run this cell after signing in to Codex.")

## 4. Load and execute the generated simulator

Run the next cell after Codex has created simulation.py. The loader checks
the benchmark interface and the required image shape before rendering.

In [ ]:
simulation_path = RUN_DIR / "simulation.py"


def load_pipeline(path: Path, config: dict):
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} does not exist. Set RUN_CODEX = True in the previous cell "
            "or point simulation_path user simulation.py."
        )
    module_name = "user_generated_" + re.sub(r"[^a-zA-Z0-9_]", "_", str(path))
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"could not import {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if not hasattr(module, "build_pipeline"):
        raise AttributeError("simulation.py must define build_pipeline(config)")
    pipeline = module.build_pipeline(config)
    if not callable(pipeline):
        raise TypeError("build_pipeline(config) must return a callable")
    return pipeline


pipeline = load_pipeline(simulation_path, config)
print(f"loaded: {simulation_path}")

In [ ]:
def image2d(value, size: int) -> np.ndarray:
    image = np.squeeze(np.asarray(value))
    if image.shape != (size, size):
        raise ValueError(
            f"simulation returned shape {image.shape}; expected {(size, size)}"
        )
    image = np.real_if_close(image)
    if np.iscomplexobj(image):
        image = np.real(image)
    return np.asarray(image, dtype=float)


def benchmark_gray(image: np.ndarray) -> np.ndarray:
    """Match the benchmark's independent 1st-to-99th percentile scaling."""
    x = np.squeeze(np.real(np.asarray(image, dtype=float)))
    finite = np.isfinite(x)
    if not finite.any():
        return np.zeros_like(x)
    lo, hi = np.percentile(x[finite], [1, 99])
    if hi <= lo:
        lo, hi = float(x[finite].min()), float(x[finite].max())
    if hi <= lo:
        return np.zeros_like(x)
    return np.clip((x - lo) / (hi - lo), 0, 1)


requested_image = image2d(pipeline(dict(particle)), int(config["image_size"]))
reuse_image = image2d(pipeline(dict(reuse_particle)), int(config["image_size"]))

np.savez_compressed(
    RUN_DIR / "images.npz",
    requested_image=requested_image,
    reuse_image=reuse_image,
)
(RUN_DIR / "reuse_particle.json").write_text(
    json.dumps(reuse_particle, indent=2) + "\n", encoding="utf-8"
)
print("requested image:", requested_image.shape, requested_image.dtype)
print("reuse image:", reuse_image.shape, reuse_image.dtype)
print(f"saved: {RUN_DIR / 'images.npz'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5), constrained_layout=True)
for ax, image, item, title in (
    (axes[0], requested_image, particle, "requested particle"),
    (axes[1], reuse_image, reuse_particle, "second particle"),
):
    ax.imshow(benchmark_gray(image), cmap="gray", interpolation="nearest")
    ax.plot(
        item["position_xy"][0],
        item["position_xy"][1],
        marker="+",
        markersize=9,
        markeredgewidth=1.25,
        color="tab:red",
    )
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(f"{MODALITY} | {ARM}")
fig.savefig(RUN_DIR / "comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"saved: {RUN_DIR / 'comparison.png'}")

## 5. Prepare all five modality requests

The following optional cell writes the five task prompts and configuration
files without calling Codex. To generate all five, review the prepared
directories and then invoke run_codex once per directory. Use a new
RUN_LABEL for another independent set of reader runs.

In [ ]:
prepared_runs = {
    modality: prepare_run(modality, ARM, RUN_LABEL)
    for modality in MODALITIES
}

for modality, item in prepared_runs.items():
    print(f"{modality:12s} -> {item['run_dir']}")

# After reviewing the prompts, an explicit all-five generation loop is:
#
# for modality, item in prepared_runs.items():
#     run_codex(item["prompt"], item["run_dir"])